# 13. Vision Transformer and detection blocks — full Swin-T topology, FPN, CenterNet objective/decode

Only tensor widths, batch size and image size are reduced. Swin-T keeps its stage depth/head/window topology; CenterNet keeps its heatmap focal objective, positive-center regression and paper-style top-K decode.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")
torch.set_num_threads(min(2, torch.get_num_threads()))
print("device:", device)

## 1. Swin-T helpers

Swin-T structural constants are preserved: patch size 4, window size 7, stage depths `[2,2,6,2]`, heads `[3,6,12,24]`, alternating W-MSA / SW-MSA inside each stage, patch merging between stages, and MLP ratio 4.

In [ ]:
def window_partition(x, window_size):
    batch_size, height, width, channels = x.shape
    pad_height = (window_size - height % window_size) % window_size
    pad_width = (window_size - width % window_size) % window_size

    x = F.pad(x, (0, 0, 0, pad_width, 0, pad_height))
    padded_height = height + pad_height
    padded_width = width + pad_width

    x = x.view(
        batch_size,
        padded_height // window_size,
        window_size,
        padded_width // window_size,
        window_size,
        channels,
    )
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    windows = windows.view(-1, window_size * window_size, channels)
    return windows, padded_height, padded_width


def window_reverse(
    windows,
    window_size,
    padded_height,
    padded_width,
    batch_size,
    original_height,
    original_width,
):
    channels = windows.size(-1)
    x = windows.view(
        batch_size,
        padded_height // window_size,
        padded_width // window_size,
        window_size,
        window_size,
        channels,
    )
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    x = x.view(batch_size, padded_height, padded_width, channels)
    return x[:, :original_height, :original_width]


def relative_position_index(window_size, device):
    coordinates = torch.stack(
        torch.meshgrid(
            torch.arange(window_size, device=device),
            torch.arange(window_size, device=device),
            indexing="ij",
        )
    ).flatten(1)
    relative = coordinates[:, :, None] - coordinates[:, None, :]
    relative = relative.permute(1, 2, 0).contiguous()
    relative[:, :, 0] += window_size - 1
    relative[:, :, 1] += window_size - 1
    relative[:, :, 0] *= 2 * window_size - 1
    return relative.sum(dim=-1)


def shifted_window_mask(
    height,
    width,
    window_size,
    shift_size,
    device,
):
    pad_height = (window_size - height % window_size) % window_size
    pad_width = (window_size - width % window_size) % window_size
    padded_height = height + pad_height
    padded_width = width + pad_width

    region = torch.zeros(
        1,
        padded_height,
        padded_width,
        1,
        device=device,
    )
    h_slices = (
        slice(0, -window_size),
        slice(-window_size, -shift_size),
        slice(-shift_size, None),
    )
    w_slices = (
        slice(0, -window_size),
        slice(-window_size, -shift_size),
        slice(-shift_size, None),
    )

    region_id = 0
    for h_slice in h_slices:
        for w_slice in w_slices:
            region[:, h_slice, w_slice, :] = region_id
            region_id += 1

    windows, _, _ = window_partition(region, window_size)
    windows = windows.squeeze(-1)
    difference = windows[:, None, :] - windows[:, :, None]
    return difference == 0

In [ ]:
class SwinWindowAttention(nn.Module):
    def __init__(
        self,
        dim,
        heads,
        window_size=7,
        shift_size=0,
    ):
        super().__init__()
        assert dim % heads == 0
        assert window_size == 7
        assert shift_size in {0, 3}

        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.window_size = window_size
        self.shift_size = shift_size

        self.qkv = nn.Linear(dim, 3 * dim)
        self.output = nn.Linear(dim, dim)
        self.relative_position_bias = nn.Parameter(
            torch.zeros((2 * window_size - 1) ** 2, heads)
        )

    def forward(self, x):
        batch_size, height, width, channels = x.shape

        if self.shift_size > 0:
            x = torch.roll(
                x,
                shifts=(-self.shift_size, -self.shift_size),
                dims=(1, 2),
            )

        windows, padded_height, padded_width = window_partition(
            x,
            self.window_size,
        )
        tokens_per_window = self.window_size ** 2

        qkv = self.qkv(windows).view(
            windows.size(0),
            tokens_per_window,
            3,
            self.heads,
            self.head_dim,
        ).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        position_index = relative_position_index(
            self.window_size,
            x.device,
        )
        position_bias = self.relative_position_bias[
            position_index.reshape(-1)
        ]
        position_bias = position_bias.view(
            tokens_per_window,
            tokens_per_window,
            self.heads,
        ).permute(2, 0, 1)
        scores = scores + position_bias[None]

        if self.shift_size > 0:
            mask = shifted_window_mask(
                height,
                width,
                self.window_size,
                self.shift_size,
                x.device,
            )
            windows_per_image = mask.size(0)
            mask = mask.repeat(batch_size, 1, 1)
            assert mask.size(0) == windows.size(0)
            scores = scores.masked_fill(
                ~mask[:, None],
                torch.finfo(scores.dtype).min,
            )

        attended = scores.softmax(dim=-1) @ v
        attended = attended.transpose(1, 2).contiguous().flatten(2)
        attended = self.output(attended)

        x = window_reverse(
            attended,
            self.window_size,
            padded_height,
            padded_width,
            batch_size,
            height,
            width,
        )

        if self.shift_size > 0:
            x = torch.roll(
                x,
                shifts=(self.shift_size, self.shift_size),
                dims=(1, 2),
            )
        return x


class SwinBlock(nn.Module):
    def __init__(self, dim, heads, shift_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = SwinWindowAttention(
            dim=dim,
            heads=heads,
            window_size=7,
            shift_size=shift_size,
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class PatchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(4 * dim)
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)

    def forward(self, x):
        batch_size, height, width, channels = x.shape
        if height % 2 or width % 2:
            x = F.pad(x, (0, 0, 0, width % 2, 0, height % 2))
            height, width = x.shape[1:3]

        x00 = x[:, 0::2, 0::2]
        x10 = x[:, 1::2, 0::2]
        x01 = x[:, 0::2, 1::2]
        x11 = x[:, 1::2, 1::2]
        merged = torch.cat([x00, x10, x01, x11], dim=-1)
        return self.reduction(self.norm(merged))


class SmallWidthSwinTiny(nn.Module):
    def __init__(self, base_dim=6, classes=10):
        super().__init__()
        self.depths = [2, 2, 6, 2]
        self.heads = [3, 6, 12, 24]
        self.window_size = 7

        self.patch_embedding = nn.Conv2d(
            3,
            base_dim,
            kernel_size=4,
            stride=4,
        )
        self.patch_norm = nn.LayerNorm(base_dim)

        stages = []
        mergers = []
        dim = base_dim
        for stage_index, (depth, heads) in enumerate(
            zip(self.depths, self.heads)
        ):
            blocks = []
            for block_index in range(depth):
                shift = 0 if block_index % 2 == 0 else 3
                blocks.append(SwinBlock(dim, heads, shift))
            stages.append(nn.ModuleList(blocks))

            if stage_index < 3:
                mergers.append(PatchMerging(dim))
                dim *= 2

        self.stages = nn.ModuleList(stages)
        self.mergers = nn.ModuleList(mergers)
        self.final_norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, classes)

    def forward(self, image):
        x = self.patch_embedding(image).permute(0, 2, 3, 1)
        x = self.patch_norm(x)

        for stage_index, blocks in enumerate(self.stages):
            for block in blocks:
                x = block(x)
            if stage_index < len(self.mergers):
                x = self.mergers[stage_index](x)

        x = self.final_norm(x).mean(dim=(1, 2))
        return self.head(x)


swin = SmallWidthSwinTiny().to(device)
assert [len(stage) for stage in swin.stages] == [2, 2, 6, 2]
assert swin.heads == [3, 6, 12, 24]
assert swin.window_size == 7
assert len(swin.mergers) == 3

swin_image = torch.randn(1, 3, 56, 56, device=device)
swin_logits = swin(swin_image)
swin_logits.square().mean().backward()
print("Swin-T stage depths:", [len(stage) for stage in swin.stages])
print("Swin-T logits:", swin_logits.shape)

## 2. FPN top-down pyramid

Four backbone levels C2–C5 are projected laterally, fused top-down, smoothed with 3×3 convolutions, and P6 is obtained from P5 by stride-2 pooling.

In [ ]:
class FeaturePyramidNetwork(nn.Module):
    def __init__(
        self,
        input_channels=(16, 24, 32, 48),
        output_channels=12,
    ):
        super().__init__()
        self.lateral = nn.ModuleList(
            [
                nn.Conv2d(channels, output_channels, 1)
                for channels in input_channels
            ]
        )
        self.smooth = nn.ModuleList(
            [
                nn.Conv2d(
                    output_channels,
                    output_channels,
                    3,
                    padding=1,
                )
                for _ in input_channels
            ]
        )

    def forward(self, features):
        assert len(features) == 4
        c2, c3, c4, c5 = features

        p5_inner = self.lateral[3](c5)
        p4_inner = self.lateral[2](c4) + F.interpolate(
            p5_inner,
            size=c4.shape[-2:],
            mode="nearest",
        )
        p3_inner = self.lateral[1](c3) + F.interpolate(
            p4_inner,
            size=c3.shape[-2:],
            mode="nearest",
        )
        p2_inner = self.lateral[0](c2) + F.interpolate(
            p3_inner,
            size=c2.shape[-2:],
            mode="nearest",
        )

        p2 = self.smooth[0](p2_inner)
        p3 = self.smooth[1](p3_inner)
        p4 = self.smooth[2](p4_inner)
        p5 = self.smooth[3](p5_inner)
        p6 = F.max_pool2d(p5, kernel_size=1, stride=2)
        return p2, p3, p4, p5, p6


fpn = FeaturePyramidNetwork().to(device)
fpn_features = (
    torch.randn(1, 16, 32, 32),
    torch.randn(1, 24, 16, 16),
    torch.randn(1, 32, 8, 8),
    torch.randn(1, 48, 4, 4),
)
fpn_outputs = fpn(fpn_features)
assert len(fpn_outputs) == 5
print("FPN shapes:", [tuple(feature.shape) for feature in fpn_outputs])

## 3. CenterNet heads, modified focal loss and positive-center regression

Heatmap supervision uses Gaussian center targets and the CenterNet modified focal objective. Offset and size losses are gathered **only at object centers**, instead of applying MSE over the full map.

In [ ]:
class CenterNetHead(nn.Module):
    def __init__(self, channels=24, classes=3):
        super().__init__()
        self.heatmap = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, classes, 1),
        )
        self.offset = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, 2, 1),
        )
        self.size = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, 2, 1),
        )

        nn.init.constant_(self.heatmap[-1].bias, -2.19)

    def forward(self, feature):
        return {
            "heatmap_logits": self.heatmap(feature),
            "offset": self.offset(feature),
            "size": self.size(feature),
        }


def gaussian2d(radius, sigma, device):
    diameter = 2 * radius + 1
    y, x = torch.meshgrid(
        torch.arange(diameter, device=device),
        torch.arange(diameter, device=device),
        indexing="ij",
    )
    center = radius
    return torch.exp(
        -((x - center) ** 2 + (y - center) ** 2)
        / (2 * sigma * sigma)
    )


def draw_gaussian(heatmap, center_x, center_y, radius=1):
    gaussian = gaussian2d(
        radius,
        sigma=max((2 * radius + 1) / 6, 1e-3),
        device=heatmap.device,
    )
    height, width = heatmap.shape

    left = min(center_x, radius)
    right = min(width - center_x - 1, radius)
    top = min(center_y, radius)
    bottom = min(height - center_y - 1, radius)

    heatmap_slice = heatmap[
        center_y - top : center_y + bottom + 1,
        center_x - left : center_x + right + 1,
    ]
    gaussian_slice = gaussian[
        radius - top : radius + bottom + 1,
        radius - left : radius + right + 1,
    ]
    torch.maximum(
        heatmap_slice,
        gaussian_slice,
        out=heatmap_slice,
    )


def centernet_focal_loss(logits, target, alpha=2.0, beta=4.0):
    probability = torch.sigmoid(logits).clamp(1e-6, 1 - 1e-6)
    positive = target.eq(1.0)
    negative = target.lt(1.0)
    negative_weight = (1 - target).pow(beta)

    positive_loss = (
        torch.log(probability)
        * (1 - probability).pow(alpha)
        * positive
    )
    negative_loss = (
        torch.log(1 - probability)
        * probability.pow(alpha)
        * negative_weight
        * negative
    )

    positive_count = positive.float().sum().clamp_min(1.0)
    return -(positive_loss.sum() + negative_loss.sum()) / positive_count


def gather_center_features(feature_map, centers):
    # centers: [B, N, 2] with (x, y)
    batch_size, channels, _, _ = feature_map.shape
    values = []
    for batch_index in range(batch_size):
        x = centers[batch_index, :, 0]
        y = centers[batch_index, :, 1]
        values.append(
            feature_map[batch_index, :, y, x].transpose(0, 1)
        )
    return torch.stack(values)


head = CenterNetHead().to(device)
features = torch.randn(2, 24, 8, 8, device=device)
prediction = head(features)

centers = torch.tensor(
    [[[4, 3]], [[2, 5]]],
    dtype=torch.long,
    device=device,
)
classes = torch.tensor([[1], [2]], device=device)
size_target = torch.tensor(
    [[[2.5, 3.0]], [[1.5, 2.0]]],
    device=device,
)
offset_target = torch.tensor(
    [[[0.25, 0.40]], [[0.10, 0.35]]],
    device=device,
)

heatmap_target = torch.zeros_like(prediction["heatmap_logits"])
for batch_index in range(2):
    x, y = centers[batch_index, 0].tolist()
    class_id = int(classes[batch_index, 0])
    draw_gaussian(
        heatmap_target[batch_index, class_id],
        x,
        y,
        radius=1,
    )

heatmap_loss = centernet_focal_loss(
    prediction["heatmap_logits"],
    heatmap_target,
)
predicted_offset = gather_center_features(
    prediction["offset"],
    centers,
)
predicted_size = gather_center_features(
    prediction["size"],
    centers,
)
offset_loss = F.l1_loss(predicted_offset, offset_target)
size_loss = F.l1_loss(predicted_size, size_target)
loss = heatmap_loss + offset_loss + 0.1 * size_loss
loss.backward()

print("CenterNet focal:", heatmap_loss.item())
print("Center regression shapes:", predicted_offset.shape, predicted_size.shape)

## 4. CenterNet local-peak + per-class top-K + global top-K decode

In [ ]:
def local_peak_nms(heatmap, kernel=3):
    padding = (kernel - 1) // 2
    local_max = F.max_pool2d(
        heatmap,
        kernel_size=kernel,
        stride=1,
        padding=padding,
    )
    return heatmap * (local_max == heatmap)


def topk_centers(heatmap, k):
    batch_size, classes, height, width = heatmap.shape
    per_class_k = min(k, height * width)

    class_scores, class_indices = torch.topk(
        heatmap.view(batch_size, classes, -1),
        per_class_k,
        dim=-1,
    )
    class_y = torch.div(
        class_indices,
        width,
        rounding_mode="floor",
    )
    class_x = class_indices % width

    final_k = min(k, classes * per_class_k)
    scores, flat_ids = torch.topk(
        class_scores.reshape(batch_size, -1),
        final_k,
        dim=-1,
    )
    class_ids = torch.div(
        flat_ids,
        per_class_k,
        rounding_mode="floor",
    )
    spatial_indices = class_indices.reshape(batch_size, -1).gather(
        1,
        flat_ids,
    )
    ys = class_y.reshape(batch_size, -1).gather(1, flat_ids)
    xs = class_x.reshape(batch_size, -1).gather(1, flat_ids)
    return scores, spatial_indices, class_ids, ys, xs


def gather_spatial(feature_map, spatial_indices):
    batch_size, channels, height, width = feature_map.shape
    flattened = feature_map.view(batch_size, channels, height * width)
    gather_index = spatial_indices[:, None].expand(-1, channels, -1)
    return flattened.gather(2, gather_index).transpose(1, 2)


def decode_centernet(prediction, k=5):
    heatmap = local_peak_nms(
        torch.sigmoid(prediction["heatmap_logits"])
    )
    scores, indices, classes, ys, xs = topk_centers(heatmap, k)
    offsets = gather_spatial(prediction["offset"], indices)
    sizes = gather_spatial(prediction["size"], indices)

    center_x = xs.float() + offsets[..., 0]
    center_y = ys.float() + offsets[..., 1]
    x1 = center_x - sizes[..., 0] / 2
    y1 = center_y - sizes[..., 1] / 2
    x2 = center_x + sizes[..., 0] / 2
    y2 = center_y + sizes[..., 1] / 2

    return {
        "scores": scores,
        "classes": classes,
        "boxes": torch.stack([x1, y1, x2, y2], dim=-1),
    }


detections = decode_centernet(head(features), k=5)
assert detections["boxes"].shape == (2, 5, 4)
print("CenterNet decoded boxes:", detections["boxes"].shape)

## Structural checklist

Assertions exercise the full Swin-T `[2,2,6,2]` stage topology with heads `[3,6,12,24]` and window 7, the four-level FPN top-down path, and CenterNet's actual modified focal + center-only L1 regression + local-peak/top-K decode. No block count or objective was replaced by a reduced surrogate.